# Cell Analysis 

## Concepts and work flow : 

Using the developed tools we will do the following for one cell : 
1. load data 
2. segment data and process number of blocks 
3. decide on KPIs to extract 
4. Extract KPIs for each cycle 
5. visualize KPIs for the cell


## Imports and Data preparation and processing 

In [12]:
import sys
import os
import yaml
import pandas as pd 


# Add src/ to sys.path so you can import batterydata
sys.path.append(os.path.abspath('../src'))

from batterydata.parsers.basytec_reader import BasyTecReader
from batterydata.pipeline.preprocessor import Preprocessor

In [2]:


with open('../local_config.yaml', 'r') as f:
    config = yaml.safe_load(f)
file_path = config['basytec_sample_file']


# Load data
reader = BasyTecReader(filepath=file_path, sample_id="demo")
result = reader.read()
df = result['data']




In [ ]:
# Instantiate Preprocessor and process
preproc = Preprocessor(df, cell_capacity_ah=1.3)
seg, doe, blocks = preproc.process_with_blocks(min_block_len=4, max_block_len=4, min_repeats=2)
tables = preproc.export_tables()
segmented_df = tables["segmented_data"]
doe_table = tables["doe_table"]
metadata_table = tables["metadata"]

In [16]:
try:
    pd.testing.assert_frame_equal(segmented_df, seg)
    print("DataFrames are exactly equal")
except AssertionError as e:
    print("DataFrames differ:", e)



DataFrames are exactly equal


In [4]:
from typing import List, Dict

def pretty_print_blocks(blocks: List[Dict[str, object]]) -> None:
    """Print a readable summary of detected blocks.

    Parameters
    ----------
    blocks : list of dict
        Output from detect_repeating_blocks_with_steps() or find_repeating_blocks().
    """
    for b in blocks:
        step = b['start']
        if b['count'] > 1:
            print(f"Step {step}: {b['count']} x ({', '.join(b['block'])})")
        else:
            print(f"Step {step}: {', '.join(b['block'])}")


In [17]:
pretty_print_blocks(blocks)

Step 0: DIS_1.00_V3.0_noCV
Step 1: OCV_600
Step 2: 200 x (CH_1.00_V4.2_CV, OCV_10, DIS_1.00_V3.0_noCV, OCV_10)
Step 802: DIS_1.00_V3.0_CV
Step 803: OCV_10
Step 804: CH_0.05_V4.2_noCV
Step 805: OCV_10
Step 806: DIS_0.05_V3.0_CV
Step 807: 160 x (OCV_10, CH_1.00_V4.2_CV, OCV_10, DIS_1.00_V3.0_noCV)
Step 1447: OCV_10
Step 1448: CH_1.00_V3.8_CV
Step 1449: 35 x (CH_1.00_V4.2_CV, OCV_10, DIS_1.00_V3.0_noCV, OCV_10)
Step 1589: CH_1.00_V4.2_CV
Step 1590: OCV_10
Step 1591: DIS_1.00_V3.9_CV


This indicates that the cell had :
1. Discharge initial step 
2. Rest for 600 seconds
3. 200 x 
    1. charge CCCV
    2. rest 10 seconds
    3. Discharge CC 
    4. rest 10 seconds 
4. Discharge 
5. qOCV step : 
    1. Charge CCCV C/20
    2. rest 10 seconds 
    3. Discharge CC C/20


## Defining KPIs 



Lets define the following KPIs to be calculated for this test procedure. 

First of all for each cycle calculate 

<ol type='a'>
<li> charge capacity</li>
<li> discharge capcity </li>
<li> Energy charge </li>
<li> Energy discharge </li>
<li> Culombic Efficiency C/E % </li>
<li> Energy Efficiency E/E % </li>
<li> Capacity retention (Qi / Q1) </li>
<li> Total charge time </li>
<li> CV step duration</li>
<li> CC step duration </li>
<li> CV duration / total charge time </li>

</ol>


In [25]:
blocks 

[{'block': ['DIS_1.00_V3.0_noCV'], 'count': 1, 'start': 0},
 {'block': ['OCV_600'], 'count': 1, 'start': 1},
 {'block': ['CH_1.00_V4.2_CV', 'OCV_10', 'DIS_1.00_V3.0_noCV', 'OCV_10'],
  'count': 200,
  'start': 2},
 {'block': ['DIS_1.00_V3.0_CV'], 'count': 1, 'start': 802},
 {'block': ['OCV_10'], 'count': 1, 'start': 803},
 {'block': ['CH_0.05_V4.2_noCV'], 'count': 1, 'start': 804},
 {'block': ['OCV_10'], 'count': 1, 'start': 805},
 {'block': ['DIS_0.05_V3.0_CV'], 'count': 1, 'start': 806},
 {'block': ['OCV_10', 'CH_1.00_V4.2_CV', 'OCV_10', 'DIS_1.00_V3.0_noCV'],
  'count': 160,
  'start': 807},
 {'block': ['OCV_10'], 'count': 1, 'start': 1447},
 {'block': ['CH_1.00_V3.8_CV'], 'count': 1, 'start': 1448},
 {'block': ['CH_1.00_V4.2_CV', 'OCV_10', 'DIS_1.00_V3.0_noCV', 'OCV_10'],
  'count': 35,
  'start': 1449},
 {'block': ['CH_1.00_V4.2_CV'], 'count': 1, 'start': 1589},
 {'block': ['OCV_10'], 'count': 1, 'start': 1590},
 {'block': ['DIS_1.00_V3.9_CV'], 'count': 1, 'start': 1591}]

In [18]:
tables.keys()

dict_keys(['segmented_data', 'doe_table', 'metadata'])

In [30]:
print(tables['doe_table']["step_sig"].iloc[:20])

0     DIS_1.00_V3.0_noCV
1                OCV_600
2        CH_1.00_V4.2_CV
3                 OCV_10
4     DIS_1.00_V3.0_noCV
5                 OCV_10
6        CH_1.00_V4.2_CV
7                 OCV_10
8     DIS_1.00_V3.0_noCV
9                 OCV_10
10       CH_1.00_V4.2_CV
11                OCV_10
12    DIS_1.00_V3.0_noCV
13                OCV_10
14       CH_1.00_V4.2_CV
15                OCV_10
16    DIS_1.00_V3.0_noCV
17                OCV_10
18       CH_1.00_V4.2_CV
19                OCV_10
Name: step_sig, dtype: object


## KPI calculation using KPI engine 


In [33]:
import pandas as pd
from batterydata.pipeline.cycle_kpi_engine import TableBundle, CycleKPIEngine, default_registry, EXAMPLE_COLUMNS

# 1) Load your dataframes
seg, doe, blocks
segmented = seg
doe       = doe
metadata  = {"chemistry": "LCO", "nominal_capacity_ah": 1.3}

# 2) Build the bundle (adjust EXAMPLE_COLUMNS if your names differ)
tables = TableBundle(segmented, doe, metadata, EXAMPLE_COLUMNS)

# 3) Create engine
engine = CycleKPIEngine(tables, default_registry())

# 4) Compute cycles 1–200 (scalar KPIs)
kpis_1_200 = engine.compute_cycles(range(1, 201))
#kpis_1_200.to_csv("artifacts/LCO_01/cycle_kpis_engine.csv", index=False)


In [34]:
kpis_1_200

,cycle_number,Q_chg_Ah,Q_dch_Ah,CE_%,E_chg_Wh,E_dch_Wh,Energy_eff_%,t_CC_s,t_CV_s,t_chg_s,t_dch_s,CV_share_%,Vavg_dch_V,Vmax_chg_V,Vmin_dch_V,Capacity_retention_%,Ah_throughput,Wh_throughput,OCV_dVdt,quality_flag
0,1,307.624434,789.194183,256.544700,969.943324,2682.352652,276.547360,69160.98492,0.0,69160.98492,82823.12082,0.0,3.398850,4.200278,2.999081,100.000000,789.194183,2682.352652,NaN,OK
1,2,518.666807,748.905069,144.390399,1610.346568,2834.667189,176.028393,0.00000,0.0,0.00000,10343.47646,NaN,3.785082,4.200278,2.998317,94.894905,1538.099252,5517.019841,NaN,OK
2,3,518.949995,747.411692,144.023836,1611.331273,2830.125561,175.638964,0.00000,0.0,0.00000,10343.04056,NaN,3.786568,4.200278,2.998126,94.705677,2285.510944,8347.145402,NaN,OK
3,4,518.648418,748.526941,144.322612,1610.227172,2833.448486,175.965760,0.00000,0.0,0.00000,10342.95484,NaN,3.785366,4.200278,2.997936,94.846992,3034.037885,11180.593887,NaN,OK
4,5,518.779685,747.239011,144.037832,1610.607653,2829.587633,175.684477,0.00000,0.0,0.00000,10342.57644,NaN,3.786724,4.200278,2.998126,94.683796,3781.276896,14010.181520,NaN,OK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,1.244288,1.243956,99.973273,4.840493,4.759106,98.318616,0.00000,0.0,0.00000,3445.06308,NaN,3.825784,4.200278,2.999462,0.157624,80014.083681,302877.426196,-0.000056,OK
196,197,1.244361,1.244002,99.971190,4.840807,4.759353,98.317341,0.00000,0.0,0.00000,3445.26240,NaN,3.825840,4.200278,2.999462,0.157629,80015.327683,302882.185549,-0.000037,OK
197,198,1.244413,1.244023,99.968640,4.841055,4.759365,98.312568,0.00000,0.0,0.00000,3445.34943,NaN,3.825785,4.200278,2.999462,0.157632,80016.571706,302886.944915,-0.000037,OK
198,199,1.244369,1.243996,99.969988,4.840816,4.759194,98.313874,0.00000,0.0,0.00000,3445.22918,NaN,3.825732,4.200278,2.999462,0.157629,80017.815702,302891.704109,-0.000075,OK
